# My way home: random Explore-Go-inspired ANN experiments

Compare standard PPO (`K=0`), fixed random prefixes, and variable random prefixes. This reuses `home_models.py`'s CNN actor?critic and Sample Factory PPO/GAE. It is a random-explorer baseline inspired by Explore-Go, **not a reproduction of the paper's learned-explorer results**.

Exploration runs inside training reset, outside `HomeImages`. Its final observation and frame history become the first policy input. No reset occurs at handover. Prefix rewards/actions never reach PPO. The native remaining episode time is retained. Evaluation always disables exploration and uses the same ordinary-reset seeds.

Keep all `explore_go_*.py`, `test_explore_go.py`, and existing `home_env.py`, `home_models.py`, `home_worker.py` alongside this notebook. Use the existing Linux environment with Sample Factory, ViZDoom, PyTorch, Gymnasium, NumPy and Matplotlib; snnTorch is not required for this ANN experiment.

In [ ]:
from pathlib import Path
import copy, json, sys, subprocess
import numpy as np

HERE = Path.cwd().resolve()
if not (HERE / "explore_go_worker.py").is_file():
    HERE = HERE / "my way home"
assert (HERE / "explore_go_worker.py").is_file(), "Start Jupyter in the repo or my way home folder"
if str(HERE) not in sys.path: sys.path.insert(0, str(HERE))
from explore_go_control import run_job, accept_smokes, check_certificate

OUTPUT_ROOT = HERE / "outputs" / "explore_go"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_TAG = "v1"                  # New tag for a new experiment; never overwrites runs.
RUN_MODE = "fresh"             # "resume" only for existing full runs with config + counters.
SMOKE_RUN_MODE = "fresh"       # Independent of full-run resume mode.
COUNTER_SAVE_SECONDS = 30.0    # Periodic snapshots; no per-action disk writes.
DEVICE = "cuda"                # A GPU allocated to you; select CUDA_VISIBLE_DEVICES before kernel startup.
STACK = 4                      # Same observation/model design for every arm.
FRAME_SKIP = 4
LEARNING_RATE = 1e-4
TRAIN_SEEDS = [0, 1, 2]
BUDGET_UNIT = "game_tics"       # Or "decisions". Includes BOTH phases; not PPO's Go-only counter.
TOTAL_BUDGET = 8_000_000        # CUMULATIVE target per run, including previous resume sessions.
EVAL_SEEDS = list(range(10_000, 10_300))
CHECKPOINT_KIND = "latest"      # Avoid selection using incomparable Go-only training returns.
ENABLE_SMOKE_TRAINING = False
ENABLE_FULL_TRAINING = False

COMMON_EXPLORE = dict(fixed_steps=50, maximum_steps=100, explore_probability=0.5,
                      maximum_episode_fraction=0.25)
CONFIGS = {
    "baseline": dict(COMMON_EXPLORE, duration="zero", explore_probability=0.0),
    "random_fixed": dict(COMMON_EXPLORE, duration="fixed"),
    "random_variable": dict(COMMON_EXPLORE, duration="uniform"),
}
CERTIFICATE = OUTPUT_ROOT / f"smoke_acceptance_{RUN_TAG}_ram.json"

def make_job(configuration, seed, smoke=False):
    return dict(root=str(OUTPUT_ROOT),
                name=f"home_eg_{configuration}_stack{STACK}_seed{seed}_{RUN_TAG}" + ("_ram_smoke" if smoke else ""),
                configuration=configuration, seed=seed, stack=STACK, frame_skip=FRAME_SKIP,
                learning_rate=LEARNING_RATE, workers=2 if smoke else 4,
                envs_per_worker=2 if smoke else 4, batch_size=128 if smoke else 1024,
                exploration=copy.deepcopy(CONFIGS[configuration]), budget_unit=BUDGET_UNIT,
                budget=32768 if smoke else TOTAL_BUDGET,
                mode=SMOKE_RUN_MODE if smoke else RUN_MODE, device=DEVICE,
                counter_save_seconds=COUNTER_SAVE_SECONDS,
                eval_seeds=EVAL_SEEDS[:10] if smoke else EVAL_SEEDS,
                checkpoint_kind=CHECKPOINT_KIND, is_smoke=smoke,
                preflight_certificate=str(CERTIFICATE))

SMOKE_JOBS = [make_job(c, 0, True) for c in CONFIGS]
JOBS = [make_job(c, seed) for seed in TRAIN_SEEDS for c in CONFIGS]
print("Kernel:", sys.executable)
print(f"Plan: {len(JOBS)} independent ANN runs; {TOTAL_BUDGET:,} total {BUDGET_UNIT} each.")
for j in JOBS: print(j["name"], j["exploration"])


## 1. Deterministic mechanism and upstream GAE tests

These test handover without an extra reset, real frame-stack history, zero exploration, mixed and capped durations, reproducibility, partial final actions, goal/timeout retries, shared counter persistence, the actual Sample Factory trajectory recorder, and upstream GAE. They use a deterministic simulator double and actual installed SF source definitions. Native training/evaluation below is a separate required gate.

In [ ]:
result = subprocess.run([sys.executable, "-m", "unittest", "-v", "test_explore_go"], cwd=HERE)
if result.returncode: raise RuntimeError("Mechanism tests failed; do not train")

## 2. Short native training and ordinary-reset evaluation

Set `ENABLE_SMOKE_TRAINING=True` above and rerun settings, then this cell. Each arm gets a short total-interaction budget, two workers, a small PPO batch, and ten ordinary-reset evaluation episodes. Smoke runs use the configured prefix settings and separate names. This checks native sampling, PPO updates, checkpoint reload, metric logging and evaluation; it does not establish learning quality.

Logs stream live and are saved under `outputs/explore_go/controller/`. Interrupting the cell stops its subprocess group. An interrupted run requires inspection, then `RUN_MODE="resume"`; use a new tag for a fresh attempt. Runs are sequential to limit GPU use.

In [ ]:
if not ENABLE_SMOKE_TRAINING:
    print("Review settings, enable smoke training, and rerun settings and this cell.")
else:
    for job in SMOKE_JOBS:
        run_job(job, "train")
        run_job(job, "evaluate")
    accept_smokes(SMOKE_JOBS, CERTIFICATE)

## 3. Full training ? only after the native smoke gate passes

Review smoke logs and summaries before setting `ENABLE_FULL_TRAINING=True`. All three arms use identical ANN architecture, PPO hyperparameters, seeds, observation settings and evaluation conditions. Only the prefix protocol differs. Ordinary resets occur with probability 0.5 in the two exploration arms; variable K is uniform over integers `[0, 100]`, so it can also produce zero-prefix episodes.

Requested K is capped at 25% of remaining decisions and leaves at least one Go decision. An exploration goal/timeout is counted, its transitions are discarded, and another native episode is attempted (bounded retries). Every such interaction still spends budget. The explorer is modular and separate from the ANN.

The budget counts actual simulator decisions or game tics, **Explore + Go**, including in-flight samples. Sample Factory's own step counter covers Go only and is reported separately. Asynchronous stopping can overshoot; compare actual totals and report overshoot. At frame skip 4, 8M tics is roughly 2M decisions, not 8M decisions. Reset initialization tics are not agent interactions. Evaluation costs are separate.

Changing source files invalidates the smoke certificate. On resume retain both checkpoint/config and the JSON counter snapshot (or legacy SQLite ledger for one-time migration). A resume is not a claim of bitwise identical asynchronous sampling.

In [ ]:
if not ENABLE_FULL_TRAINING:
    print("Full training is disabled. Complete smoke training/evaluation and review results first.")
else:
    check_certificate(CERTIFICATE)
    for job in JOBS:
        run_job(job, "train")

## 4. Matched evaluation with exploration disabled

Run after full training. Every model starts from ordinary native resets, with deterministic argmax actions, identical episode seeds, original task rewards and the native episode timeout. No explorer, warm-up, shortened episode, or hidden prefix is used. The latest checkpoint avoids choosing between arms using differently distributed Go-phase training returns. These evaluation seeds are for development; reserve a separate held-out set for final reporting.

In [ ]:
for job in JOBS:
    run_job(job, "evaluate")

## 5. Scorecard and charts

Primary comparison: ordinary-reset goal success rate and original episode return. Report episode length alongside success: a short failure is not efficient navigation. Training curves show Go-phase return and PPO diagnostics; Go returns across different starting-state distributions do not replace ordinary-reset evaluation. Counts include exploration successes, timeouts, valid handovers and total interactions.

In [ ]:
import matplotlib.pyplot as plt
import csv
reports = []
for job in JOBS:
    folder = OUTPUT_ROOT / "results" / job["name"]
    path = folder / f"evaluation_{CHECKPOINT_KIND}" / "summary.json"
    if not path.exists():
        print("Not evaluated:", job["name"]); continue
    report = json.loads(path.read_text())
    counts = report["actual_training"]["actual"]
    reports.append(dict(configuration=job["configuration"], seed=job["seed"],
                        success=report["goal_success_rate"], mean_return=report["mean_return"],
                        decisions=report["mean_episode_decisions"],
                        episode_seconds=report["mean_episode_game_seconds"],
                        explore_decisions=counts.get("explore_decisions",0), go_decisions=counts.get("go_decisions",0),
                        total_game_tics=counts["total_game_tics"], total_decisions=counts["total_decisions"],
                        exploration_goals=counts.get("explore_goal",0),
                        exploration_timeouts=counts.get("explore_timeout",0),
                        valid_go_starts=counts.get("valid_go_starts",0),
                        explored_go_starts=counts.get("explored_go_starts",0),
                        training_seconds=report["total_training_seconds"]))
if reports:
    with (OUTPUT_ROOT / "comparison.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(reports[0])); writer.writeheader(); writer.writerows(reports)
    print(f"{'Configuration':20} {'Seed':>4} {'Success':>9} {'Return':>9} {'Decisions':>10} {'Explore':>10} {'Go':>10}")
    for r in reports:
        print(f"{r['configuration']:20} {r['seed']:4d} {r['success']:9.3f} {r['mean_return']:9.4f} {r['decisions']:10.1f} {r['explore_decisions']:10.0f} {r['go_decisions']:10.0f}")
    fig, axes = plt.subplots(1, 3, figsize=(15,4))
    for ax, metric, label in zip(axes, ['success','mean_return','decisions'], ['Ordinary-reset success rate','Original episode return','Mean episode decisions']):
        groups = [[r[metric] for r in reports if r['configuration']==c] for c in CONFIGS]
        ax.bar(list(CONFIGS), [np.mean(v) if v else np.nan for v in groups],
               yerr=[np.std(v,ddof=1) if len(v)>1 else 0 for v in groups], capsize=4)
        for i, vals in enumerate(groups): ax.scatter([i]*len(vals), vals, color='black', zorder=3)
        ax.set_title(label); ax.tick_params(axis='x', rotation=20)
    fig.suptitle('Dots: independent training seeds; error bars: SD across training seeds')
    fig.tight_layout(); fig.savefig(OUTPUT_ROOT/'evaluation_comparison.png', dpi=150); plt.show()

fig, axes = plt.subplots(1,3,figsize=(16,4))
for job in JOBS:
    path = OUTPUT_ROOT/'results'/job['name']/'training_metrics.jsonl'
    if not path.exists(): continue
    rows = [json.loads(line) for line in path.read_text().splitlines()]
    x = [r['total_'+BUDGET_UNIT] for r in rows]
    for ax, key in zip(axes,['go_mean_return','entropy','value_loss']):
        y = [(r.get('go_completed_return',0)/r['go_episodes'] if r.get('go_episodes',0)>0 else np.nan)
             if key=='go_mean_return' else r.get(key,np.nan) for r in rows]
        ax.plot(x,y,label=f"{job['configuration']} seed{job['seed']}")
        ax.set_xlabel('Total Explore + Go '+BUDGET_UNIT); ax.set_title(key)
axes[0].legend(fontsize=6); fig.tight_layout()
fig.savefig(OUTPUT_ROOT/'training_curves.png',dpi=150); plt.show()

## 6. Aggregate result and next decision

Compare configurations across **all** planned training seeds, not the luckiest checkpoint. Rank first by mean ordinary-reset success rate, then mean original return. This is a development comparison, not proof of statistical significance or guaranteed improvement. No best configuration is declared until all planned runs have evaluation results. Inspect replay/behavior before launching a larger study.

In [ ]:
if len(reports) != len(JOBS):
    print(f"Incomplete comparison: {len(reports)}/{len(JOBS)} evaluated runs. No winner declared.")
else:
    aggregates = []
    for configuration in CONFIGS:
        group = [r for r in reports if r['configuration']==configuration]
        success = [r['success'] for r in group]
        aggregates.append(dict(configuration=configuration, mean_success=float(np.mean(success)),
                               std_across_training_seeds=float(np.std(success,ddof=1)) if len(success)>1 else None,
                               mean_return=float(np.mean([r['mean_return'] for r in group]))))
    aggregates.sort(key=lambda r:(r['mean_success'],r['mean_return']),reverse=True)
    print(json.dumps(aggregates,indent=2))
    print('Best development configuration:',aggregates[0]['configuration'])
    (OUTPUT_ROOT/'aggregate_comparison.json').write_text(json.dumps(aggregates,indent=2))

## Boundary and accounting notes

- PPO and GAE are Sample Factory's implementation. A rollout ending mid-Go bootstraps from its next observation. A goal or the native finite 2,100-tic task horizon ends the task and does not bootstrap into another episode. Handover creates no artificial done or transition. `with_vtrace=False`, `value_bootstrap=False`; unsupported external continuing-task truncations fail visibly rather than silently use an incorrect terminal value. This notebook does not support arbitrary artificial time-limit wrappers.
- Standard SF random decorrelation steps are disabled for **all** arms to prevent an uncounted second exploration mechanism. CNN/model code and PPO losses are unchanged.
- Tic accounting uses native `my_way_home` reward (`goal bonus + living_reward ? tics`) and checks the clock when available. This handles terminal clock resets and partial final actions. It rejects changed reward schemes. Do not add reward shaping to these experiments.
- Workers update shared RAM counters without per-action disk writes. The parent saves atomic JSON snapshots every `COUNTER_SAVE_SECONDS` (default 30 seconds) and at orderly shutdown. Live budgets use exact RAM counts; forced termination can lose work since the last successful save, so crash/resume totals may undercount. Aggregate exploration/Go outcomes and reset ordinals are retained; per-reset event rows are no longer stored. Periodic snapshots can stay on NFS; GPU contention and checkpoint/log I/O may still limit speed. Stop old training before migrating: resume reads its old SQLite counts once, leaves the original database intact, and subsequently uses JSON. New source requires fresh `_ram_smoke` validation before resuming a full run.
- `training_sessions.json` retains timing/count history across resumes; a session's `seconds` is not total lifetime training time. All outputs are separate from your existing ANN/SNN runs.
- Random Explore-Go may fail to improve exploration, and terminal prefixes can waste budget. Use the outcome counts and ordinary-reset scorecard to decide whether it helps before attempting a learned explorer.